## let us add the user information first

In [ ]:
customer_name = "Anand"
amount = 65000
complaint = "My laptop arrived damaged"

## Conditional Statements 

--> we wil be able to make descision based on certain consitions

In [ ]:
if amount <=10000:
    print("Automatic review and approval")
else: 
    print("Manager review")    

## Functions

A fucntion we will have a reuseable block of a code

In [ ]:
def check_amount(amount):
    if amount <= 10000:
        return "Automatic review"
    return "Manager review"

## Dictionaries
- This concept becomes important because when  we introduce the LangGrapgh state

In [ ]:
customer = {
    "name": "Anand",
    "amount": 65000,
    "complaint": "My laptop arrived damaged"
}

print(f"Customer Name: {customer['name']}")
print(f"Amount: {customer['amount']}")
print(f"Complaint: {customer['complaint']}")

## Pydantic

Pydantic is a python library for defining data models and validating input.

A business application should not blindly trust the information submitted through a form.
For example, a customer might enter a negative amount, omit their name, or submit an extremely short complaint.


 - BaseModel -> It gives the class validation capabilities.
 - Field(min_length=2) -> requires at least 2 chaaracters
 - Field(gt=0) -> the number must be greater than zero
 When we create a Complaint object, pydantic will validate the supplied values

In [ ]:
from pydantic import BaseModel, Field

class Complaint(BaseModel):
    customer_name: str = Field(min_length=2)
    amount: float = Field(gt=0)
    complaint: str = Field(min_length=20)

unlike the rules based mechanisms LLM can interpret many different ways of expressing the same problem

For E.g,
- "My laptop screen is broken"
- "The diplay arrived crack"
- "I opened the package and found the screen damanged"

Customer meesage -> Tokenization -> Transformer model processes context -> Predict and generate output tokens -> Complaint analysis

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template(
    "Analyze this customer complaint: {complaint}"
)

## Model integration

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0
)

## Chains

- A chain connects components into execution pipeline.
- The output of first function becomes the input for the later immediate function
- This | symbol is an operator which comes fromt the LCEL - LangChain Expressin language, this comes from Runnables

Prompt template -> language model -> model response


In [ ]:
chain = prompt | llm 

## Invoking the chain

In [ ]:
result = chain.invoke({"complaint": "My laptop arrived damaged"})

print(result.content)

## Structured output from the LLM

- Complaint category
- Customer sentiment
- Requesed action
- complaint summary


The relevant schema would be created using Pydantic - why? because instead of recieving a long paragraph we defie a pydantic schema

In [ ]:
class ComplainAnalysis(BaseModel):
    category: str
    sentiment: str
    requested_action: str
    summary: str

In [ ]:
structured_llm = llm.with_structured_output(
    ComplainAnalysis
)

analysis_chain = prompt | structured_llm

print(analysis_chain.invoke({"complaint": "My laptop arrived damaged"}))

{'category': 'Product Damage', 'sentiment': 'Negative', 'requested_action': 'Request replacement or repair of the damaged laptop', 'summary': 'The customer received a laptop that was damaged upon arrival.'}


/var/folders/nz/73gn1yv53rbbbrxz2hsly4hm0000gn/T/ipykernel_28209/3983431733.py:7: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.14/migration/
  print(analysis_chain.invoke({"complaint": "My laptop arrived damaged"}).dict())


In [ ]:
{'category': 'Product Damage', 'sentiment': 'Negative', 'requested_action': 'Request replacement or repair of the damaged laptop', 'summary': 'The customer received a laptop that was damaged upon arrival.'}

## LangGraph State